# sqd-go — Quickstart: derive state with a custom processor

This builds a real indexer: as LBTC `Transfer` events stream in, a **custom processor**
maintains a per-address position (total in / out / transfer count) in a ClickHouse table
you define. You will:

1. install Go, the `sqd-go` CLI, and ClickHouse,
2. scaffold a project (`config.yaml` + `custom_schema.go` + `custom_processor.go`),
3. index a range with `--state` (which compiles your processor in), and
4. query both the raw events **and** the derived per-address state.

Along the way we cover the two things that trip up newcomers: **how Go modules work here**
and **proto vs. `--no-proto` mode**.

## 1. Install Go

`sqd-go` is a Go program **and** a code generator: it generates Go for your indexer and
compiles it, so a working Go toolchain is required. This installs Go into `/root/.go`.

In [ ]:
# Install Go (~30s)
!wget -q -O - https://raw.githubusercontent.com/canha/golang-tools-install-script/master/goinstall.sh | bash >/dev/null 2>&1

In [ ]:
import os
# Make Go and any `go install`-ed binaries (like sqd-go) visible to every shell cell.
os.environ['GOROOT'] = '/root/.go'
os.environ['GOPATH'] = '/root/go'
os.environ['PATH']   = '/root/.go/bin:/root/go/bin:' + os.environ['PATH']
!go version

## 2. Install the sqd-go CLI

The install script runs `go install github.com/franz101/sqd-go@latest`, dropping the
`sqd-go` binary into `$(go env GOPATH)/bin` (already on `PATH` from the cell above).

In [ ]:
# Install the CLI (first run compiles dependencies, ~1-2 min)
!curl -sSL https://raw.githubusercontent.com/franz101/sqd-go/main/install.sh | bash
!sqd-go help | head -20

## 3. Go concepts for beginners (optional but recommended)

If you're new to Go, here are the key concepts you'll encounter in this tutorial:

### Go Packages and Modules
- **Package**: A folder of Go files sharing a `package` declaration (e.g., `package main`)
- **Module**: A collection of packages with a `go.mod` file that manages dependencies
- **Import**: How you include code from other packages: `import "fmt"` or `import "github.com/franz101/sqd-go/sqd"`

### Go Types and Structs
- **Struct**: Groups data together: `type User struct { Name string; Age int }`
- **Pointer**: References memory location: `var p *User = &User{Name: "Alice"}`
- **Interface**: Defines behavior: `type Event interface { Meta() EventMeta }`

### Go Functions and Methods
- **Function**: Standalone code: `func add(a, b int) int { return a + b }`
- **Method**: Function attached to a type: `func (u User) Greet() string { return "Hi " + u.Name }`
- **init()**: Special function that runs automatically at program startup

### Go Error Handling
- **Explicit errors**: Functions return `result, error` - no exceptions!
- **Check errors**: Always check `if err != nil { return err }`
- **nil means success**: When `err == nil`, the operation succeeded

### Go Syntax Sugar
- **Short declaration**: `name := "Alice"` (type inferred automatically)
- **Multiple return**: `return result, err` (functions can return multiple values)
- **Range loops**: `for i, item := range slice { ... }` (iterate over collections)
- **Type assertions**: `val, ok := event.(*Transfer)` (check concrete type)

---

## 4. How Go modules work here (read this — it prevents the #1 failure)

Your indexer **is its own Go module**. When you run `sqd-go start <proj> --state`, the CLI
scaffolds a `go.mod` for your project that depends on `github.com/franz101/sqd-go`, runs
codegen, then `go build`s and runs it.

Two rules follow, and breaking either is the most common cause of a broken project:

1. **One package name per directory.** `custom_schema.go` and `custom_processor.go` must
   declare the *same* `package` (Go errors with *“found packages X and Y”* otherwise).
2. **Import the public facade, never `internal/`.** Your processor imports
   `github.com/franz101/sqd-go/sqd` (the public API) and your own `"<module>/generated"`
   package — never the module's `internal/...` packages (Go forbids importing another
   module's `internal/` tree). Full explanation: [`docs/GO_MODULES.md`](GO_MODULES.md).

**Want to learn more?** See [`docs/GO_FOR_BEGINNERS.md`](GO_FOR_BEGINNERS.md) for a detailed Go tutorial.

## 3. Run ClickHouse

sqd-go indexes into [ClickHouse](https://clickhouse.com). In Colab we use the official
single binary with an **empty password** on the default ports (native `9000`, HTTP `8123`).

In [ ]:
# Download the ClickHouse single binary
!curl -s https://clickhouse.com/ | sh >/dev/null 2>&1

# Start the server in the background (empty password) and give it a moment to come up
import subprocess, time
subprocess.Popen(['./clickhouse', 'server'],
                 stdout=open('clickhouse-server.log','w'),
                 stderr=subprocess.STDOUT)
time.sleep(8)
!./clickhouse client --query "SELECT 'ClickHouse is up' AS status"

## 5. Scaffold the project

Same `config.yaml` as the minimal demo — chain, range, contract, and the `Transfer` event.

In [ ]:
!mkdir -p lbtcdemo

In [ ]:
%%writefile lbtcdemo/config.yaml
name: lbtcdemo
chains:
  - id: 1
    start_block: 20600000
    end_block: 20660000          # ~60k blocks so derived state commits during the run
    contracts:
      - name: LBTC
        address: "0x8236a87084f8B84306f72007F36F2618A5634494"
        events:
          - event: Transfer(address indexed from, address indexed to, uint256 value)

## 6. Define the derived state (`custom_schema.go`)

Each struct ending in **`Schema`** becomes a hot-state entity. The suffix is stripped, so
`UserPositionSchema` → `state.UserPosition` (with `.Get(key)` / `.Save(value, meta)`), and
the `// pk:` comment names the primary key.

Every event your processor sees also carries **always-present metadata** — `BlockNumber`,
`BlockTimestamp`, `TransactionIndex`, `LogIndex`, ... — see [`docs/EVENT_FIELDS.md`](EVENT_FIELDS.md).
`Save()` automatically stamps `UpdatedAtBlock` / `UpdatedAt` from that metadata.

### Go concepts in this file:
- **`package lbtcdemo`**: All Go files in a directory must share the same package name
- **`import`**: Loads external packages (time, common.Address for Ethereum addresses, uint256.Int for large numbers)
- **`type ... struct`**: Defines a data structure with named fields
- **`// pk: Address`**: Special comment that marks which field is the primary key
- **Field types**: `common.Address` (20-byte Ethereum address), `uint256.Int` (very large numbers for amounts), `uint64` (regular integers), `time.Time` (timestamps)

**Why these types?**
- `common.Address`: For Ethereum wallet addresses (not regular strings!)
- `uint256.Int`: Token amounts can be huge (up to 2^256 - 1), bigger than regular integers
- `uint64`: For counts and block numbers (0 to 18,446,744,073,709,551,615)
- `time.Time`: Go's built-in time type with timezone support

In [ ]:
%%writefile lbtcdemo/custom_schema.go
package lbtcdemo

import (
	"time"

	"github.com/ethereum/go-ethereum/common"
	"github.com/holiman/uint256"
)

// pk: Address
type UserPositionSchema struct {
	Address        common.Address // primary key: the account
	TotalIn        uint256.Int    // cumulative LBTC received
	TotalOut       uint256.Int    // cumulative LBTC sent
	TransferCount  uint64         // transfers touching this account
	UpdatedAtBlock uint64         // set automatically by Save()
	UpdatedAt      time.Time      // set automatically by Save()
}

## 7. Write the processor (`custom_processor.go`)

`Process(state, block)` runs once per parsed block. Iterate `block.EventsIter()` and
type-switch on the generated event structs (`Transfer` on contract `LBTC` →
`*generated.LBTCTransfer`). Register it in `init()` **under `generated.ProjectName`** so the
registered name always matches your config.

Note the import comments — the module-relative `generated` import and the public `sqd` facade.

### Go concepts in this file:
- **`func Process(...) error`**: Function that takes parameters and returns an error (nil = success)
- **`*generated.State`**: Pointer to State (efficient for large structs, allows mutation)
- **`for ... range`**: Go's iteration loop over collections
- **`ev.(*generated.LBTCTransfer)`**: Type assertion to check if event is Transfer type
- **`, ok` pattern**: Comma-ok idiom for safe type assertions (ok = true if successful)
- **`state.UserPosition.Get(...)`**: Method call to look up existing state
- **`&generated.UserPosition{...}`**: Get address of new struct (creates pointer)
- **`pos.TotalOut.Add(...)`**: Method call that modifies value in place (uint256.Int is mutable)
- **`state.UserPosition.Save(...)`**: Persist updated state to ClickHouse
- **`func init()`**: Special function that runs automatically at program startup
- **`func() (...) { ... }`**: Anonymous function (closure) returned as value

### Key patterns to understand:
1. **Zero address check**: `var zero common.Address` creates the null/zero address (all bytes = 0)
2. **Type assertion**: `e, ok := ev.(*generated.LBTCTransfer)` safely checks event type
3. **Get-or-create**: `pos, ok := state.UserPosition.Get(key); if !ok { pos = &... }` loads existing or creates new
4. **In-place mutation**: `pos.TotalOut.Add(&pos.TotalOut, &e.Value)` modifies existing value
5. **State persistence**: `state.UserPosition.Save(pos, e.EventMeta)` marks state for ClickHouse write

**Why `e.EventMeta`?** Events carry metadata (block number, timestamp, transaction hash) that ClickHouse needs for ordering and time-travel queries.

In [ ]:
%%writefile lbtcdemo/custom_processor.go
package lbtcdemo

import (
	"github.com/ethereum/go-ethereum/common"

	// Your OWN generated package. Import path = "<module>/generated".
	// `sqd-go` writes it during --state; don't edit it by hand.
	generated "lbtcdemo/generated"

	// Public facade. Import this, never the module's internal/ packages,
	// so the project can build as its own module.
	"github.com/franz101/sqd-go/sqd"
)

func Process(state *generated.State, block *generated.ParsedBlock) error {
	var zero common.Address
	for ev := range block.EventsIter() {
		e, ok := ev.(*generated.LBTCTransfer)
		if !ok {
			continue
		}
		if e.From != zero { // debit sender
			pos, ok := state.UserPosition.Get(e.From)
			if !ok {
				pos = &generated.UserPosition{Address: e.From}
			}
			pos.TotalOut.Add(&pos.TotalOut, &e.Value)
			pos.TransferCount++
			state.UserPosition.Save(pos, e.EventMeta)
		}
		if e.To != zero { // credit receiver
			pos, ok := state.UserPosition.Get(e.To)
			if !ok {
				pos = &generated.UserPosition{Address: e.To}
			}
			pos.TotalIn.Add(&pos.TotalIn, &e.Value)
			pos.TransferCount++
			state.UserPosition.Save(pos, e.EventMeta)
		}
	}
	return nil
}

func init() {
	generated.CustomProcessFn = Process
	sqd.RegisterProcessor(generated.ProjectName, func() (sqd.Processor, error) {
		return generated.NewProcessor(sqd.GetProtoMode())
	})
}

## 8. Index with `--state`

`--state` is what makes your processor run: it regenerates the project, **compiles your
`Process` into a fresh binary**, and execs it. (Plain `start` uses the prebuilt CLI, whose
processor registry is empty — it would index raw events but skip your custom logic.)

We pass **`--no-proto`**. The single-function `Process(state, block)` API runs in the
default (V1) decode path; proto mode is an advanced performance mode that needs a separate
`ProcessProto`. If you forget `--no-proto` here, sqd-go now **fails loudly** telling you so
(rather than silently leaving the derived tables empty).

In [ ]:
!CLICKHOUSE_HOST=127.0.0.1 \
 CLICKHOUSE_NATIVE_PORT=9000 CLICKHOUSE_HTTP_PORT=8123 \
 CLICKHOUSE_USER=default CLICKHOUSE_PASSWORD='' \
 sqd-go start lbtcdemo --start-block 20600000 --end-block 20660000 --restart --parallel-fetch --state --no-proto

## 9. Query the derived state

`lbtc_transfer_events` holds the raw events; `user_positions` holds the per-address state your
processor built. Query `user_positions` with `FINAL` to collapse to the latest row per key.

In [ ]:
!./clickhouse client --query "SHOW TABLES FROM lbtcdemo"

In [ ]:
!./clickhouse client --query "SELECT count() AS raw_transfers FROM lbtcdemo.lbtc_transfer_events"
!./clickhouse client --query "SELECT count() AS positions FROM lbtcdemo.user_positions FINAL"

In [ ]:
# Top accounts by total received (FINAL collapses to the latest row per key)
!./clickhouse client --query "SELECT concat('0x', lower(hex(address))) AS account, total_in, total_out, transfer_count, updated_at_block FROM lbtcdemo.user_positions FINAL ORDER BY total_in DESC LIMIT 10 FORMAT PrettyCompact"

## 10. Metrics & tuning

While indexing, sqd-go prints a `stats` line (throughput in `blk/s`, events, checkpoint) and
a `profile` line (time in fetch / parse / insert / custom) every ~10s. Knobs like
`SQD_PARALLEL_FETCHERS`, `SQD_PARALLEL_PAGE_SIZE`, and `SQD_COLDCACHE_MB` tune throughput and
memory; CPU profiling is `--cpuprofile cpu.prof`. Full reference: [`docs/METRICS.md`](METRICS.md).

## Troubleshooting

### Common Go-Specific Errors

| Error Message | What It Means | How to Fix |
| --- | --- | --- |
| `found packages X and Y in directory` | Different package names in same directory | Ensure both files use `package lbtcdemo` (or your project name) |
| `use of internal package ... not allowed` | Trying to import `internal/` from another module | Import `github.com/franz101/sqd-go/sqd` instead |
| `undefined: generated.LBTCTransfer` | Type doesn't exist in generated code | Check event name in config.yaml matches (case-sensitive!) |
| `cannot use ... as type ...` | Type mismatch in assignment or function call | Check types match (e.g., `common.Address` vs `string`) |
| `expected ..., found ...` | Wrong number or type of function arguments | Count and types must match function signature |
| `syntax error: unexpected ...` | Missing semicolon, brace, or other syntax error | Usually missing `}` or `;` - check line above error |

### Common sqd-go Issues

| Symptom | Cause | Fix |
| --- | --- | --- |
| `found packages X and Y` | `custom_schema.go` / `custom_processor.go` use different `package` names | Use one package name |
| `use of internal package not allowed` | imported `github.com/franz101/sqd-go/internal/...` | Import `.../sqd` instead |
| `undefined: generated.SomethingTransfer` | event isn't in `config.yaml`, or wrong contract/event name | The type is `<Contract><Event>`; add the event and re-run |
| derived tables empty + a loud error about proto mode | ran `--state` without `--no-proto` | add `--no-proto` |
| derived tables empty, run looks fine, **no** error | ran plain `start` (empty registry) | add `--state` |
| start block ignored | used `--start-block=N` (the `=` form) | use space form: `--start-block N` |
| `cannot find package ...` | Import path is wrong or dependency missing | Check import path matches module name, run `go mod tidy` |

### Debugging Tips for Go Beginners

1. **Read error messages carefully**: Go errors are detailed and tell you exactly where the problem is
2. **Check line numbers**: Go errors include file:line - look there first
3. **Use `go fmt`**: Auto-format your code: `go fmt custom_processor.go`
4. **Use `go vet`**: Find additional issues: `go vet ./...`
5. **Check imports**: Ensure all imported packages are actually used
6. **Match package names**: All files in same directory must use same package name
7. **Pointer vs value**: Check if function expects `*Type` (pointer) or `Type` (value)
8. **Exported names**: Only upper-case names (e.g., `Process`) are visible outside package

### Getting Help

- **Documentation**: [`docs/GO_FOR_BEGINNERS.md`](docs/GO_FOR_BEGINNERS.md) - Go concepts tutorial
- **Modules**: [`docs/GO_MODULES.md`](docs/GO_MODULES.md) - How Go modules work in sqd-go
- **Events**: [`docs/EVENT_FIELDS.md`](docs/EVENT_FIELDS.md) - Standard event fields
- **Metrics**: [`docs/METRICS.md`](docs/METRICS.md) - Performance monitoring
- **Examples**: `examples/` directory - Working indexers you can study

**Still stuck?** Check that your `package` names match, imports are correct, and you're using the right types. Most errors are simple typos or misunderstandings about Go's type system!